In [1]:
import os
import sys
import random
import numpy as np
import pandas as pd

# --- Fix: prevent TF/protobuf crash by forcing pure-Python protobuf BEFORE any TF/protobuf import.
# This addresses: AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'
os.environ["PROTOCOL_BUFFERS_PYTHON_IMPLEMENTATION"] = "python"
os.environ.setdefault("PROTOCOL_BUFFERS_PYTHON_IMPLEMENTATION_VERSION", "2")
os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "2")
os.environ.setdefault("TF_FORCE_GPU_ALLOW_GROWTH", "true")


# Block importing protobuf's C++ extension which triggers the incompatibility in some Kaggle images.
class _BlockProtobufCxxExtImporter:
    """Blocks google.protobuf.pyext._message (C++ ext) so TF uses pure-Python protobuf."""

    def find_spec(self, fullname, path=None, target=None):
        if fullname in ("google.protobuf.pyext", "google.protobuf.pyext._message"):
            raise ImportError(
                f"Blocked import of {fullname} to avoid protobuf C++ runtime incompatibility"
            )
        return None


_blocker = _BlockProtobufCxxExtImporter()
if not any(isinstance(x, _BlockProtobufCxxExtImporter) for x in sys.meta_path):
    sys.meta_path.insert(0, _blocker)

# Purge any already-loaded protobuf modules (must happen before importing TF).
for k in list(sys.modules.keys()):
    if k.startswith("google.protobuf"):
        del sys.modules[k]

# Extra guard: pre-seed the problematic modules as "missing" so import resolution fails fast.
sys.modules["google.protobuf.pyext"] = None
sys.modules["google.protobuf.pyext._message"] = None

SEED = 42
os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)

import pydicom

try:
    import cv2

    def _resize2d(arr, out_hw):
        out_h, out_w = out_hw
        return cv2.resize(arr, (out_w, out_h), interpolation=cv2.INTER_AREA).astype(
            np.float32
        )

except Exception:
    cv2 = None

    def _resize2d(arr, out_hw):
        out_h, out_w = out_hw
        in_h, in_w = arr.shape[:2]
        y_idx = (np.linspace(0, in_h - 1, out_h)).astype(np.int32)
        x_idx = (np.linspace(0, in_w - 1, out_w)).astype(np.int32)
        return arr[y_idx][:, x_idx].astype(np.float32)


def _try_import_tf(seed=SEED):
    # --- Fix: import tensorflow only inside this function, after protobuf guards are installed.
    try:
        import tensorflow as tf
        from tensorflow.keras import layers

        tf.random.set_seed(seed)
        print("TF version:", tf.__version__)
        return True, tf, layers, tf.keras
    except Exception as e:
        print(
            "WARNING: TensorFlow failed to import; will write a valid fallback submission.csv"
        )
        print("TF import error:", repr(e))
        return False, None, None, None


TF_AVAILABLE, tf, layers, keras = _try_import_tf()

# Always define to avoid NameError downstream even if TF import fails.
common_test_ids = []
p_final = np.array([], dtype=np.float32)



2026-05-12 23:12:31.888404: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1778627551.923727      11 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1778627551.932956      11 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

TF version: 2.18.0


In [2]:
DATA_ROOT = "/kaggle/input/rsna-miccai-brain-tumor-radiogenomic-classification"
TRAIN_DIR = os.path.join(DATA_ROOT, "train")
TEST_DIR = os.path.join(DATA_ROOT, "test")
LABELS_CSV = os.path.join(DATA_ROOT, "train_labels.csv")
SAMPLE_SUB = os.path.join(DATA_ROOT, "sample_submission.csv")

assert os.path.exists(TEST_DIR), f"Missing TEST_DIR: {TEST_DIR}"
assert os.path.exists(SAMPLE_SUB), f"Missing SAMPLE_SUB: {SAMPLE_SUB}"

sample_sub = pd.read_csv(SAMPLE_SUB)
print(sample_sub.shape, sample_sub.columns.tolist())

if TF_AVAILABLE:
    assert os.path.exists(TRAIN_DIR), f"Missing TRAIN_DIR: {TRAIN_DIR}"
    assert os.path.exists(LABELS_CSV), f"Missing LABELS_CSV: {LABELS_CSV}"
    labels_df = pd.read_csv(LABELS_CSV)
    print(labels_df.shape, labels_df.columns.tolist())



(59, 2) ['BraTS21ID', 'MGMT_value']
(526, 2) ['BraTS21ID', 'MGMT_value']


In [3]:
IMG_PX_SIZE = 150
SLICES_PER_CASE = 6


def _sorted_case_dirs(path_root):
    return sorted([f.path for f in os.scandir(path_root) if f.is_dir()])


def _safe_case_id_from_path(case_dir):
    return os.path.basename(case_dir)


def _load_case_series_slices(
    case_dir, series_name, img_px_size=IMG_PX_SIZE, slices_per_case=SLICES_PER_CASE
):
    series_dir = os.path.join(case_dir, series_name)
    if not os.path.isdir(series_dir):
        return None  # missing series

    dcm_files = sorted(
        [
            f.path
            for f in os.scandir(series_dir)
            if f.is_file() and f.name.lower().endswith(".dcm")
        ]
    )
    if len(dcm_files) == 0:
        return None

    out = []
    count = 0
    for fp in dcm_files:
        try:
            ds = pydicom.dcmread(fp)
            arr = ds.pixel_array.astype(np.float32)
        except Exception:
            continue

        if arr.sum() <= 100000:
            continue

        arr_r = _resize2d(arr, (img_px_size, img_px_size))
        stacked = np.stack([arr_r, arr_r, arr_r], axis=-1)
        mx = float(np.max(stacked))
        if mx <= 0:
            continue
        stacked_norm = stacked / mx

        if stacked_norm.sum() <= 2000:
            continue

        out.append(stacked_norm.astype(np.float32))
        count += 1
        if count >= slices_per_case:
            break

    if len(out) < slices_per_case:
        if len(out) == 0:
            pad = np.zeros((img_px_size, img_px_size, 3), dtype=np.float32)
            out = [pad] * slices_per_case
        else:
            out = out + [out[-1]] * (slices_per_case - len(out))

    return np.stack(out, axis=0).astype(np.float32)  # (S, H, W, C)


def load_dataset_slices(path_root, series_name, case_ids_filter=None):
    case_dirs = _sorted_case_dirs(path_root)
    X = []
    case_ids = []
    for case_dir in case_dirs:
        cid = _safe_case_id_from_path(case_dir)
        if case_ids_filter is not None and cid not in case_ids_filter:
            continue
        vol = _load_case_series_slices(case_dir, series_name)
        if vol is None:
            continue
        X.append(vol)
        case_ids.append(cid)
    if len(X) == 0:
        return None, []
    X = np.stack(X, axis=0)  # (N, S, H, W, C)
    return X, case_ids




In [4]:
if TF_AVAILABLE:
    bad_cases = {"00109", "00123", "00709"}
    labels_df["BraTS21ID"] = labels_df["BraTS21ID"].astype(str).str.zfill(5)
    labels_df = labels_df[~labels_df["BraTS21ID"].isin(bad_cases)].reset_index(
        drop=True
    )

    train_case_ids = set(labels_df["BraTS21ID"].tolist())

    X_t2_train, t2_train_ids = load_dataset_slices(
        TRAIN_DIR, "T2w", case_ids_filter=train_case_ids
    )
    X_fl_train, fl_train_ids = load_dataset_slices(
        TRAIN_DIR, "FLAIR", case_ids_filter=train_case_ids
    )

    if X_t2_train is None or X_fl_train is None:
        raise RuntimeError(
            "Failed to load training data for T2w/FLAIR. Please verify dataset integrity/paths."
        )

    common_train_ids = sorted(set(t2_train_ids).intersection(set(fl_train_ids)))
    id_to_label = dict(
        zip(labels_df["BraTS21ID"].values, labels_df["MGMT_value"].values)
    )

    def _filter_by_ids(X, ids, keep_ids):
        keep_set = set(keep_ids)
        idx = [i for i, cid in enumerate(ids) if cid in keep_set]
        Xf = X[idx]
        idsf = [ids[i] for i in idx]
        return Xf, idsf

    X_t2_train, t2_train_ids = _filter_by_ids(
        X_t2_train, t2_train_ids, common_train_ids
    )
    X_fl_train, fl_train_ids = _filter_by_ids(
        X_fl_train, fl_train_ids, common_train_ids
    )

    y_train = np.array([id_to_label[cid] for cid in common_train_ids], dtype=np.float32)

    print(
        "Train T2:",
        X_t2_train.shape,
        "Train FLAIR:",
        X_fl_train.shape,
        "y:",
        y_train.shape,
    )

    X_t2_test, t2_test_ids = load_dataset_slices(TEST_DIR, "T2w", case_ids_filter=None)
    X_fl_test, fl_test_ids = load_dataset_slices(
        TEST_DIR, "FLAIR", case_ids_filter=None
    )

    if X_t2_test is None:
        X_t2_test = np.zeros(
            (0, SLICES_PER_CASE, IMG_PX_SIZE, IMG_PX_SIZE, 3), dtype=np.float32
        )
        t2_test_ids = []
    if X_fl_test is None:
        X_fl_test = np.zeros(
            (0, SLICES_PER_CASE, IMG_PX_SIZE, IMG_PX_SIZE, 3), dtype=np.float32
        )
        fl_test_ids = []

    common_test_ids = sorted(set(t2_test_ids).intersection(set(fl_test_ids)))
    X_t2_test, t2_test_ids = _filter_by_ids(X_t2_test, t2_test_ids, common_test_ids)
    X_fl_test, fl_test_ids = _filter_by_ids(X_fl_test, fl_test_ids, common_test_ids)

    print("Test T2:", X_t2_test.shape, "Test FLAIR:", X_fl_test.shape)
    print("Example test IDs:", common_test_ids[:5])



Train T2: (523, 6, 150, 150, 3) Train FLAIR: (523, 6, 150, 150, 3) y: (523,)
Test T2: (59, 6, 150, 150, 3) Test FLAIR: (59, 6, 150, 150, 3)
Example test IDs: ['00002', '00019', '00021', '00031', '00054']


In [5]:
if TF_AVAILABLE:
    S = SLICES_PER_CASE
    H = IMG_PX_SIZE
    W = IMG_PX_SIZE
    C = 3

    def flatten_slices(X_case, y_case=None):
        N = X_case.shape[0]
        Xs = X_case.reshape(N * S, H, W, C)
        if y_case is None:
            return Xs
        ys = np.repeat(y_case, S).astype(np.float32)
        return Xs, ys

    X_t2_s, y_s = flatten_slices(X_t2_train, y_train)
    X_fl_s, _ = flatten_slices(X_fl_train, y_train)

    print("Per-slice T2:", X_t2_s.shape, "Per-slice y:", y_s.shape)



Per-slice T2: (3138, 150, 150, 3) Per-slice y: (3138,)


In [6]:
if TF_AVAILABLE:

    def build_model(input_shape=(IMG_PX_SIZE, IMG_PX_SIZE, 3)):
        inp = keras.Input(shape=input_shape)
        x = layers.Conv2D(16, 3, padding="same", activation="relu")(inp)
        x = layers.MaxPooling2D()(x)
        x = layers.Conv2D(32, 3, padding="same", activation="relu")(x)
        x = layers.MaxPooling2D()(x)
        x = layers.Conv2D(64, 3, padding="same", activation="relu")(x)
        x = layers.GlobalAveragePooling2D()(x)
        x = layers.Dense(64, activation="relu")(x)
        x = layers.Dropout(0.2)(x)
        out = layers.Dense(1, activation="sigmoid")(x)
        model = keras.Model(inp, out)
        model.compile(optimizer=keras.optimizers.Adam(1e-3), loss="binary_crossentropy")
        return model

    N_slices = X_t2_s.shape[0]
    rng = np.random.RandomState(SEED)
    idx = np.arange(N_slices)
    rng.shuffle(idx)
    val_frac = 0.15
    n_val = max(1, int(N_slices * val_frac))
    val_idx = idx[:n_val]
    tr_idx = idx[n_val:]

    X_t2_tr, y_tr = X_t2_s[tr_idx], y_s[tr_idx]
    X_t2_va, y_va = X_t2_s[val_idx], y_s[val_idx]
    X_fl_tr = X_fl_s[tr_idx]
    X_fl_va = X_fl_s[val_idx]

    model_t2 = build_model()
    model_fl = build_model()

    BATCH = 32
    EPOCHS = 2

    history_t2 = model_t2.fit(X_t2_tr, y_tr, batch_size=BATCH, epochs=EPOCHS, verbose=1)
    history_fl = model_fl.fit(X_fl_tr, y_tr, batch_size=BATCH, epochs=EPOCHS, verbose=1)

    t2_tr_mean = float(model_t2.predict(X_t2_tr, batch_size=64, verbose=0).mean())
    t2_va_mean = float(model_t2.predict(X_t2_va, batch_size=64, verbose=0).mean())
    fl_tr_mean = float(model_fl.predict(X_fl_tr, batch_size=64, verbose=0).mean())
    fl_va_mean = float(model_fl.predict(X_fl_va, batch_size=64, verbose=0).mean())
    print("Pred mean T2 train/val:", t2_tr_mean, t2_va_mean)
    print("Pred mean FL train/val:", fl_tr_mean, fl_va_mean)



2026-05-12 23:13:08.395968: E external/local_xla/xla/stream_executor/cuda/cuda_driver.cc:152] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


Epoch 1/2
84/84 ━━━━━━━━━━━━━━━━━━━━ 7s 59ms/step - loss: 0.6925
Epoch 2/2
84/84 ━━━━━━━━━━━━━━━━━━━━ 5s 57ms/step - loss: 0.6915
Epoch 1/2
84/84 ━━━━━━━━━━━━━━━━━━━━ 6s 58ms/step - loss: 0.6919
Epoch 2/2
84/84 ━━━━━━━━━━━━━━━━━━━━ 5s 58ms/step - loss: 0.6920
Pred mean T2 train/val: 0.5148382186889648 0.5148400664329529
Pred mean FL train/val: 0.5120090246200562 0.5119957327842712


In [7]:
if TF_AVAILABLE:

    def predict_casewise(model, X_case):
        N = X_case.shape[0]
        if N == 0:
            return np.zeros((0,), dtype=np.float32)
        Xs = X_case.reshape(N * S, H, W, C)
        ps = model.predict(Xs, batch_size=64, verbose=0).reshape(N, S)
        return ps.mean(axis=1)

    p_t2 = predict_casewise(model_t2, X_t2_test)
    p_fl = predict_casewise(model_fl, X_fl_test)

    p_final = 0.5 * p_t2 + 0.5 * p_fl

    overall_mean = 0.25 * (t2_tr_mean + t2_va_mean + fl_tr_mean + fl_va_mean)
    p_final = 0.9 * p_final + 0.1 * overall_mean

    p_final = np.clip(p_final, 1e-6, 1 - 1e-6)

    if p_final.shape[0] > 0:
        print(
            "Pred stats:",
            float(p_final.min()),
            float(p_final.max()),
            float(p_final.mean()),
        )
    else:
        print(
            "Pred stats: no common test cases loaded; will default to 0.5 in submission."
        )

sub = sample_sub.copy()
sub["BraTS21ID"] = sub["BraTS21ID"].astype(str).str.zfill(5)

if TF_AVAILABLE:
    pred_map = dict(zip(common_test_ids, p_final.astype(float)))
    sub["MGMT_value"] = sub["BraTS21ID"].map(pred_map).fillna(0.5).astype(float)
else:
    sub["MGMT_value"] = 0.5

sub.to_csv("submission.csv", index=False)
print("Wrote submission.csv with shape:", sub.shape)
print(sub.head())

Pred stats: 0.5130487680435181 0.5140005350112915 0.5135270953178406
Wrote submission.csv with shape: (59, 2)
  BraTS21ID  MGMT_value
0     00356    0.513193
1     00140    0.514001
2     00757    0.513638
3     00275    0.513223
4     00570    0.513590
